<a href="https://colab.research.google.com/github/Yeeejj/EMHA/blob/feat/select-v4/notebooks/colab_select_v4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# EMHA v4: exercise selection (EXPLORATORY, Deviation 7)

Bootstrap only: clone, install, set paths, call `src.training.run_select_v4`.
All logic and settings live in `src/` (`run_select_v4.py`, `SelectV4Config` in
`src/utils/config.py`). Plan: `DOCS/plans/v4_exercise_selection.txt`.

**CPU only.** Use a CPU runtime (no GPU units needed). v4 reads v3's outputs
from `DRIVE_OUTPUT_ROOT` (handcrafted features, embeddings, v3
`predictions.csv`, v3 draws) and writes only v4 paths. Needs the Colab secret
`GITHUB_TOKEN` with notebook access switched on.

**Branch or patch.** If `feat/select-v4` is on GitHub, it is cloned. If it is
not pushed yet, `main` is cloned and the v4 commits are applied from
`PATCH_FILE` on Drive (`git am`), so the same code runs either way.

**Easiest: `MODE = "all"`** (the default). One Run all does everything in
order: snapshot (skipped if it already exists), preflight, rank, combine,
evaluate, draws, permutation, report, compare. It stops at the first error.
Rank folds and null draws are cached on Drive, so if the session drops, just
Run all again with the same settings and it picks up where it stopped.

**Other modes** (`MODE`):
- `"snapshot"`: SHA-256 of `results/` and `models/` to `SNAPSHOT_FILE`, once,
  before any v4 step. Refuses if the file exists.
- `"compare"`: same tree vs `SNAPSHOT_FILE`; new files allowed only under v4
  paths (`ALLOW_NEW`). Must print `OK`.
- `"run"`: one step only, `python -m src.training.run_select_v4 --step STEP`.

Every command is logged to `logs/v4_<name>_<UTC>.log` on Drive.

Do not open any v4 output before the report step finishes. Keep the tab
open while it runs (about 2-3 hours on a free CPU runtime, mostly the 20 null
draws).

In [ ]:
MODE = "all"  # "all" | "run" | "snapshot" | "compare"
STEP = "preflight"  # preflight | rank | combine | evaluate | draws | permutation | report
DRAWS = None  # e.g. "0-9" (STEP = "draws" only); None = all
N_JOBS = -1
BRANCH = "feat/select-v4"
FALLBACK_BRANCH = "main"
PATCH_FILE = "/content/drive/MyDrive/EMHA/v4_select.patch"  # used only if BRANCH is not on GitHub
DRIVE_ZIP = "/content/drive/MyDrive/EMHA/emha-crops.zip"
LOCAL_DATA_ROOT = "/content/emha-crops"
DRIVE_OUTPUT_ROOT = "/content/drive/MyDrive/EMHA/run_outputs"
SNAPSHOT_FILE = DRIVE_OUTPUT_ROOT + "/v4_checksums_before.txt"
TOPS = ["results", "models"]
ALLOW_NEW = [
    "results/select_v4/",
    "results/final/permutation_runs/primary_v4/",
    "results/final/EXPLORATORY_v4.txt",
    "results/final/fig_v4_",
]

REPO = "Yeeejj/EMHA"
REPO_DIR = "/content/EMHA"

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
from google.colab import userdata

token = userdata.get("GITHUB_TOKEN")

import base64
import os
import subprocess

auth = base64.b64encode(f"x-access-token:{token}".encode()).decode()
GIT = ["git", "-c", f"http.extraHeader=Authorization: Basic {auth}"]
url = f"https://github.com/{REPO}.git"
if os.path.isdir(REPO_DIR):
    print(f"{REPO_DIR} already exists; using it as is")
else:
    heads = subprocess.run(GIT + ["ls-remote", "--heads", url, BRANCH],
                           capture_output=True, text=True, check=True).stdout
    if heads.strip():
        subprocess.run(GIT + ["clone", "--branch", BRANCH, url, REPO_DIR], check=True)
    else:
        if not os.path.isfile(PATCH_FILE):
            raise SystemExit(f"{BRANCH} is not on GitHub and {PATCH_FILE} is missing")
        print(f"{BRANCH} not on GitHub; cloning {FALLBACK_BRANCH} and applying {PATCH_FILE}")
        subprocess.run(GIT + ["clone", "--branch", FALLBACK_BRANCH, url, REPO_DIR], check=True)
        subprocess.run(["git", "checkout", "-b", BRANCH], cwd=REPO_DIR, check=True)
        subprocess.run(["git", "-c", "user.name=colab", "-c", "user.email=colab@localhost",
                        "am", PATCH_FILE], cwd=REPO_DIR, check=True)
del auth, GIT, token
os.chdir(REPO_DIR)
branch = subprocess.run(["git", "rev-parse", "--abbrev-ref", "HEAD"],
                        capture_output=True, text=True, check=True).stdout.strip()
if branch != BRANCH:
    raise SystemExit(f"{REPO_DIR} is on {branch!r}, not {BRANCH}; delete it and Run all")
print(subprocess.run(["git", "log", "--oneline", "-3"], capture_output=True,
                     text=True).stdout)

In [ ]:
%pip install -q -r requirements.txt

In [ ]:
!git rev-parse HEAD
!git describe --tags --always
!git tag --list protocol-frozen
!nproc
import torch

print("torch", torch.__version__, "| CUDA available", torch.cuda.is_available())

In [ ]:
import shutil

if MODE in ("run", "all"):
    labels_csv = os.path.join(LOCAL_DATA_ROOT, "metadata", "labels.csv")
    if not os.path.isfile(labels_csv):
        if not os.path.isfile(DRIVE_ZIP):
            raise SystemExit(f"{DRIVE_ZIP} not found; upload emha-crops.zip to Drive")
        partial = LOCAL_DATA_ROOT + ".partial"
        shutil.rmtree(partial, ignore_errors=True)
        print("unzipping", DRIVE_ZIP, "->", LOCAL_DATA_ROOT)
        shutil.unpack_archive(DRIVE_ZIP, partial, "zip")
        os.rename(partial, LOCAL_DATA_ROOT)
    if not os.path.isfile(labels_csv):
        raise SystemExit(f"{LOCAL_DATA_ROOT} has no metadata/labels.csv; check the zip")
    os.environ["EMHA_DATA_ROOT"] = LOCAL_DATA_ROOT
    os.environ["EMHA_OUTPUT_ROOT"] = DRIVE_OUTPUT_ROOT
    os.environ.update({"OMP_NUM_THREADS": "1", "MKL_NUM_THREADS": "1",
                       "OPENBLAS_NUM_THREADS": "1"})
os.makedirs(os.path.join(DRIVE_OUTPUT_ROOT, "logs"), exist_ok=True)
print({"MODE": MODE, "STEP": STEP, "DRAWS": DRAWS, "N_JOBS": N_JOBS})

In [ ]:
import datetime
import sys

PY = [sys.executable, "-u", "-m"]
SNAPSHOT = PY + ["scripts.hash_tree", "snapshot", DRIVE_OUTPUT_ROOT, "--tops", *TOPS,
                 "--out", SNAPSHOT_FILE]
COMPARE = PY + ["scripts.hash_tree", "compare", DRIVE_OUTPUT_ROOT, "--before",
                SNAPSHOT_FILE, "--tops", *TOPS, "--allow-new", *ALLOW_NEW]
ALL_STEPS = ["preflight", "rank", "combine", "evaluate", "draws", "permutation", "report"]
STEP_CMD = {
    s: PY + ["src.training.run_select_v4", "--step", s, "--n-jobs", str(N_JOBS)]
    + (["--draws", DRAWS] if DRAWS and s == "draws" else [])
    for s in ALL_STEPS
}

if MODE == "snapshot":
    if os.path.exists(SNAPSHOT_FILE):
        raise SystemExit(f"{SNAPSHOT_FILE} already exists; snapshots are never overwritten")
    cmds = [("snapshot", SNAPSHOT)]
elif MODE == "compare":
    cmds = [("compare", COMPARE)]
elif MODE == "run":
    cmds = [(STEP, STEP_CMD[STEP])]
elif MODE == "all":
    cmds = [] if os.path.exists(SNAPSHOT_FILE) else [("snapshot", SNAPSHOT)]
    if not cmds:
        print("snapshot already exists; skipping it (resumed run)")
    cmds += [(s, STEP_CMD[s]) for s in ALL_STEPS] + [("compare", COMPARE)]
else:
    raise SystemExit(f"MODE must be all, run, snapshot or compare, not {MODE!r}")

head = subprocess.run(["git", "rev-parse", "HEAD"], cwd=REPO_DIR,
                      capture_output=True, text=True, check=True).stdout.strip()
stamp = datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%dT%H%M%SZ")
name = STEP if MODE == "run" else MODE
log_path = os.path.join(DRIVE_OUTPUT_ROOT, "logs", f"v4_{name}_{stamp}.log")
print("log:", log_path)
run_exit, failed = 0, None
with open(log_path, "a", buffering=1) as log:
    log.write(f"git HEAD {head} | nproc {os.cpu_count()}\n")
    for label, cmd in cmds:
        banner = f"\n===== {label} ====="
        print(banner)
        log.write(banner + "\ncommand: " + " ".join(cmd) + "\n")
        proc = subprocess.Popen(cmd, cwd=REPO_DIR, stdout=subprocess.PIPE,
                                stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in proc.stdout:
            print(line, end="")
            log.write(line)
        run_exit = proc.wait()
        log.write(f"{label} exit code: {run_exit}\n")
        if run_exit != 0:
            failed = label
            break
drive.flush_and_unmount()
print("\nDrive flushed and unmounted; check the outputs on Drive before disconnecting")
if failed:
    raise SystemExit(f"{failed} failed (exit {run_exit}); fix it, then Run all again")
print(f"{name}: all done")
if MODE == "all":
    print("Report: results/final/EXPLORATORY_v4.txt under", DRIVE_OUTPUT_ROOT)